# RMIT RAG Evaluation — Retrieval + DeepEval + Readability + Robustness (Out of KB and Abstentions)

This notebook covers:

- **Retrieval:** Hit@3, Recall@3, NDCG@3
- **DeepEval:** Answer Relevancy + Faithfulness
- **Readability:** Flesch Reading Ease + Flesch-Kincaid Grade Level + Gunning Fog Index + SMOG Index + answer complexity measures
- DeepEval sample: **4 per persona = 12 questions in total**


## Evaluation Approach

To evaluate the RAG system, we assess both **retrieval quality** and **generated answer quality**. This is important because a RAG system can fail at different stages: it may retrieve the wrong evidence, or it may retrieve the correct evidence but generate an inaccurate or unsupported answer.

### Retrieval Metrics

Three complementary retrieval metrics are used:

- **Hit@3** measures whether at least one relevant passage appears within the top three retrieved results. This provides a simple indication of whether the retriever is able to surface useful evidence for a query.

- **Recall@3** measures the proportion of all relevant passages that are retrieved within the top three results. This is useful for identifying cases where the retriever finds some relevant evidence but misses other important supporting passages.

- **NDCG@3** evaluates both relevance and ranking position. Relevant passages receive greater credit when they appear higher in the retrieved results, making this metric useful for assessing whether the most useful evidence is prioritised.

These metrics were chosen together because they capture different aspects of retrieval performance: **Hit@3 measures presence, Recall@3 measures coverage, and NDCG@3 measures ranking quality**.

### Generation Metrics

For answer generation, **DeepEval** is used to evaluate:

- **Answer Relevancy** — whether the generated response directly addresses the user's question.
- **Faithfulness** — whether the generated response is supported by the retrieved context rather than introducing unsupported information.

DeepEval was selected because these metrics align closely with the main goals of a RAG system: responses should be both **relevant to the user's query** and **grounded in the retrieved source material**.

Using DeepEval alongside retrieval metrics allows the system to be evaluated as a complete RAG pipeline rather than assessing retrieval or generation in isolation. This distinction is important because strong retrieval performance does not necessarily guarantee a correct final answer, and a well-written answer may still be unreliable if it is not supported by the retrieved evidence.

### Answer Complexity (Readability) Metrics

For generated answers, readability metrics are used to evaluate the complexity and accessibility of responses for a reader:

* **Flesch Reading Ease** — estimates how easy the response is to read. Higher scores indicate easier-to-read text.
* **Flesch-Kincaid Grade Level** — estimates the US school grade level required to understand the response. Lower scores indicate simpler text.
* **Gunning Fog Index** — estimates the number of years of formal education generally needed to understand the response on a first reading. Lower scores indicate simpler text and fewer complex words.
* **SMOG Index** — estimates the education level needed to understand a response based on the frequency of complex, multi-syllable words. Lower scores indicate simpler text. As chatbot answers are often relatively short, SMOG scores should be interpreted cautiously.
* **Average words per sentence** — measures sentence length and provides an additional indicator of answer complexity. Longer sentences may indicate more complex sentence structures.
* **Average syllables per word** — measures word complexity at a basic lexical level. Higher values generally indicate the use of longer or more complex words.

These metrics complement **Answer Relevancy** and **Faithfulness** by evaluating the **presentation, readability, and accessibility** of generated answers rather than whether the answers are relevant or factually supported. They are descriptive measures and should not be interpreted as definitive assessments of answer quality or suitability for a particular reader.


In [8]:
import os
import re
import time
import numpy as np
import pandas as pd
import ollama

from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 140)

TOPICS_FILE = "topics_WIL20.csv"
PASSAGES_FILE = "passages_WIL20.csv"
QRELS_FILE = "qrels_WIL20.txt"

GENERATOR_MODEL = "llama3.2:3b"
JUDGE_MODEL = "qwen3:1.7b"

TOP_K = 3
DEEPEVAL_PER_PERSONA = 4
RANDOM_STATE = 42

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "180"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "1"


## Load data

In [9]:
topics = pd.read_csv(TOPICS_FILE)
passages_df = pd.read_csv(PASSAGES_FILE)

qrels = pd.read_csv(
    QRELS_FILE,
    sep=r"\s+",
    header=None,
    names=["question_id", "unused", "passage_id", "relevance"]
)

print("Topics:", topics.shape)
print("Passages:", passages_df.shape)
print("Qrels:", qrels.shape)

display(topics.head())
display(passages_df.head())
display(qrels.head())


Topics: (70, 6)
Passages: (45, 4)
Qrels: (72, 4)


,topic_id,topic,question_id,question,persona,status
0,C01,Can I study a Bachelor of Business online?,C01Q01,Can I study a Bachelor of Business online?,prospective student,known
1,C01,Can I study a Bachelor of Business online?,C01Q02,Is there an online version of the Business degree?,current student,known
2,C02,What career outcomes does the Marketing major lead to?,C02Q01,What career outcomes does the Marketing major lead to?,parent/guardian,known
3,C02,What career outcomes does the Marketing major lead to?,C02Q02,What jobs can students expect after majoring in Marketing?,current student,known
4,C03,Can I enrol in advanced electives early?,C03Q01,Am I eligible to take a third-year elective as a first-year student?,current student,known


,passage_id,passage,school,program
0,P01,"The Bachelor of Business is available to study online at RMIT. The full-time, online Bachelor of Business takes 36 months to complete an...","Economics, Finance and Marketing",Bachelor of Business
1,P02,"The Marketing major prepares graduates for roles in digital marketing, brand management, campaign strategy, and customer analytics acros...","Economics, Finance and Marketing",Bachelor of Business
2,P03,"At RMIT, you can take a third-year elective as a first-year student as long as you meet the course requirements and your program structu...","Economics, Finance and Marketing",General
3,P04,"When you successfully complete this degree, you may be eligible for entry into a range of RMIT Honours and postgraduate qualifications i...","Economics, Finance and Marketing",Bachelor of Commerce
4,P05,"The Bachelor of Graphic Design offers four majors: Branding, Experience Design, Illustration, and Typography. The Branding major focuses...",Design,Bachelor of Graphic Design


,question_id,unused,passage_id,relevance
0,C01Q01,0,P01,2
1,C01Q01,0,P15,1
2,C01Q02,0,P01,2
3,C01Q02,0,P15,1
4,C02Q01,0,P02,2


# Retrieval

In [10]:
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

tokenised_passages = [
    tokenize(text)
    for text in passages_df["passage"].fillna("").tolist()
]

bm25 = BM25Okapi(tokenised_passages)

def retrieve_top_k(question, k=TOP_K):
    scores = bm25.get_scores(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "rank": rank,
            "passage_id": passages_df.iloc[i]["passage_id"],
            "passage": passages_df.iloc[i]["passage"],
            "score": float(scores[i]),
        }
        for rank, i in enumerate(top_indices, start=1)
    ]


In [11]:
qrels_by_question = {}

for question_id, group in qrels.groupby("question_id"):
    qrels_by_question[question_id] = dict(
        zip(group["passage_id"], group["relevance"])
    )

def dcg(relevances):
    return sum(
        (2 ** rel - 1) / np.log2(rank + 2)
        for rank, rel in enumerate(relevances)
    )

def retrieval_metrics_for_question(
    question_id,
    question,
    retriever,
    k=TOP_K
):
    judged = qrels_by_question.get(question_id)

    if not judged:
        return None

    retrieved = retriever(question, k=k)
    retrieved_ids = [
        x["passage_id"]
        for x in retrieved
    ]

    relevant_ids = {
        pid for pid, rel in judged.items()
        if rel > 0
    }

    hit = int(
        any(pid in relevant_ids for pid in retrieved_ids)
    )

    recall = (
        len(set(retrieved_ids) & relevant_ids)
        / len(relevant_ids)
        if relevant_ids else np.nan
    )

    retrieved_rels = [
        int(judged.get(pid, 0))
        for pid in retrieved_ids
    ]

    ideal_rels = sorted(
        [int(rel) for rel in judged.values()],
        reverse=True
    )[:k]

    retrieved_rels += [0] * (
        k - len(retrieved_rels)
    )

    ideal_rels += [0] * (
        k - len(ideal_rels)
    )

    ideal_dcg = dcg(ideal_rels)

    ndcg = (
        dcg(retrieved_rels) / ideal_dcg
        if ideal_dcg > 0
        else np.nan
    )

    return {
        "retrieved_passage_ids": retrieved_ids,
        f"hit@{k}": hit,
        f"recall@{k}": recall,
        f"ndcg@{k}": ndcg,
    }

In [14]:
retrieval_rows = []

for _, row in topics.iterrows():
    metrics = retrieval_metrics_for_question(
        row["question_id"],
        row["question"],
        retriever=retrieve_top_k,
        k=TOP_K
    )

    if metrics is not None:
        retrieval_rows.append({
            "question_id": row["question_id"],
            "persona": row["persona"],
            "question": row["question"],
            **metrics
        })

retrieval_results = pd.DataFrame(retrieval_rows)

display(retrieval_results)

,question_id,persona,question,retrieved_passage_ids,hit@3,recall@3,ndcg@3
0,C01Q01,prospective student,Can I study a Bachelor of Business online?,"[P01, P16, P15]",1,1.0,0.963940
1,C01Q02,current student,Is there an online version of the Business degree?,"[P01, P16, P44]",1,0.5,0.826235
2,C02Q01,parent/guardian,What career outcomes does the Marketing major lead to?,"[P02, P32, P14]",1,1.0,1.000000
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"[P02, P04, P44]",1,1.0,1.000000
4,C03Q01,current student,Am I eligible to take a third-year elective as a first-year student?,"[P03, P19, P12]",1,1.0,1.000000
5,C03Q02,current student,Can I enrol in advanced electives early?,"[P03, P09, P11]",1,1.0,1.000000
6,C04Q01,parent/guardian,Does the Bachelor of Commerce prepare students for further postgraduate study?,"[P44, P20, P38]",0,0.0,0.000000
7,C04Q02,current student,Can I go on to a Master's after this degree?,"[P04, P44, P10]",1,1.0,1.000000
8,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?,"[P40, P06, P41]",0,0.0,0.000000
9,C05Q02,prospective student,What study areas does Bachelor of Graphic Design offer?,"[P20, P32, P40]",0,0.0,0.000000


In [15]:
retrieval_summary = pd.DataFrame({
    "metric": [f"Hit@{TOP_K}", f"Recall@{TOP_K}", f"NDCG@{TOP_K}"],
    "score": [
        retrieval_results[f"hit@{TOP_K}"].mean(),
        retrieval_results[f"recall@{TOP_K}"].mean(),
        retrieval_results[f"ndcg@{TOP_K}"].mean(),
    ]
})

display(retrieval_summary.round(3))


,metric,score
0,Hit@3,0.804
1,Recall@3,0.735
2,NDCG@3,0.706


In [16]:
retrieval_by_persona = (
    retrieval_results
    .groupby("persona")[
        [f"hit@{TOP_K}", f"recall@{TOP_K}", f"ndcg@{TOP_K}"]
    ]
    .mean()
    .round(3)
)

display(retrieval_by_persona)


,hit@3,recall@3,ndcg@3
persona,,,
current student,0.947,0.868,0.850
parent/guardian,0.667,0.625,0.596
prospective student,0.750,0.675,0.636


In [17]:
hit_failures = retrieval_results[
    retrieval_results[f"hit@{TOP_K}"] == 0
].copy()

print("Hit@3 failures:", len(hit_failures))

display(
    hit_failures[
        ["question_id", "persona", "question", "retrieved_passage_ids"]
    ]
)


Hit@3 failures: 10


,question_id,persona,question,retrieved_passage_ids
6,C04Q01,parent/guardian,Does the Bachelor of Commerce prepare students for further postgraduate study?,"[P44, P20, P38]"
8,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?,"[P40, P06, P41]"
9,C05Q02,prospective student,What study areas does Bachelor of Graphic Design offer?,"[P20, P32, P40]"
10,C06Q01,prospective student,What career opportunities are available after completing the Bachelor of Graphic Design?,"[P44, P23, P40]"
12,C07Q01,prospective student,What does the Bachelor of Games focus on?,"[P24, P32, P39]"
13,C07Q02,current student,What areas will I study in the Bachelor of Games?,"[P20, P35, P43]"
23,C14Q01,prospective student,What software should I be expected to learn during the Bachelor of Games?,"[P16, P17, P44]"
32,C23Q01,parent/guardian,Will students get any industry experience while completing Commerce?,"[P33, P25, P20]"
33,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,"[P14, P32, P44]"
49,C50Q01,parent/guardian,What careers can my child pursue after completing the Bachelor of Games?,"[P44, P41, P06]"


In [18]:
retrieval_results.to_csv("retrieval_evaluation_results.csv", index=False)
retrieval_summary.to_csv("retrieval_evaluation_summary.csv", index=False)
retrieval_by_persona.to_csv("retrieval_evaluation_by_persona.csv")

print("Retrieval results saved.")


Retrieval results saved.


## Semantic and Hybrid Retrieval

In [19]:
# Semantic retrieval using Ollama embeddings

EMBED_MODEL = "nomic-embed-text"

def get_embedding(text):
    response = ollama.embeddings(
        model=EMBED_MODEL,
        prompt=str(text)
    )

    return np.array(
        response["embedding"],
        dtype=np.float32
    )


# Embed each KB passage once
passage_embeddings = np.vstack([
    get_embedding(text)
    for text in passages_df["passage"].fillna("")
])

print("Embedding model:", EMBED_MODEL)
print("Passage embedding matrix:", passage_embeddings.shape)

Embedding model: nomic-embed-text
Passage embedding matrix: (45, 768)


In [20]:
def cosine_similarity(query_vector, passage_matrix):
    query_norm = query_vector / np.linalg.norm(query_vector)

    passage_norms = passage_matrix / np.linalg.norm(
        passage_matrix,
        axis=1,
        keepdims=True
    )

    return passage_norms @ query_norm


def retrieve_semantic(question, k=TOP_K):
    query_embedding = get_embedding(question)

    scores = cosine_similarity(
        query_embedding,
        passage_embeddings
    )

    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "rank": rank,
            "passage_id": passages_df.iloc[i]["passage_id"],
            "passage": passages_df.iloc[i]["passage"],
            "score": float(scores[i]),
        }
        for rank, i in enumerate(top_indices, start=1)
    ]

In [21]:
def retrieve_hybrid(question, k=TOP_K, candidate_k=10):
    bm25_results = retrieve_top_k(
        question,
        k=candidate_k
    )

    semantic_results = retrieve_semantic(
        question,
        k=candidate_k
    )

    rrf_scores = {}
    passage_lookup = {}

    # Reciprocal Rank Fusion
    for results in [bm25_results, semantic_results]:
        for rank, result in enumerate(results, start=1):
            pid = result["passage_id"]

            rrf_scores[pid] = (
                rrf_scores.get(pid, 0)
                + 1 / (60 + rank)
            )

            passage_lookup[pid] = result

    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )

    final_results = []

    for rank, pid in enumerate(ranked_ids[:k], start=1):
        final_results.append({
            "rank": rank,
            "passage_id": pid,
            "passage": passage_lookup[pid]["passage"],
            "score": rrf_scores[pid],
        })

    return final_results

In [22]:
retrievers = {
    "BM25": retrieve_top_k,
    "Semantic": retrieve_semantic,
    "Hybrid": retrieve_hybrid
}

comparison_rows = []

for name, retriever in retrievers.items():

    results = []

    for _, row in topics.iterrows():
        metrics = retrieval_metrics_for_question(
            question_id=row["question_id"],
            question=row["question"],
            retriever=retriever,
            k=TOP_K
        )

        if metrics is not None:
            results.append(metrics)

    results_df = pd.DataFrame(results)

    comparison_rows.append({
        "retriever": name,
        "n": len(results_df),
        "hit@3": results_df["hit@3"].mean(),
        "recall@3": results_df["recall@3"].mean(),
        "ndcg@3": results_df["ndcg@3"].mean()
    })

retrieval_comparison = pd.DataFrame(comparison_rows)

display(retrieval_comparison)

,retriever,n,hit@3,recall@3,ndcg@3
0,BM25,51,0.803922,0.735294,0.706352
1,Semantic,51,0.882353,0.843137,0.781130
2,Hybrid,51,0.843137,0.784314,0.767302


In [23]:
comparison_details = []

for _, row in topics.iterrows():
    qid = row["question_id"]

    # Only evaluate questions that have relevance judgements
    if qid not in qrels_by_question:
        continue

    bm25_metrics = retrieval_metrics_for_question(
        qid,
        row["question"],
        retrieve_top_k,
        k=TOP_K
    )

    semantic_metrics = retrieval_metrics_for_question(
        qid,
        row["question"],
        retrieve_semantic,
        k=TOP_K
    )

    comparison_details.append({
        "question_id": qid,
        "persona": row["persona"],
        "question": row["question"],

        "bm25_hit": bm25_metrics["hit@3"],
        "semantic_hit": semantic_metrics["hit@3"],

        "bm25_recall": bm25_metrics["recall@3"],
        "semantic_recall": semantic_metrics["recall@3"],

        "bm25_ndcg": bm25_metrics["ndcg@3"],
        "semantic_ndcg": semantic_metrics["ndcg@3"],

        "bm25_passages": bm25_metrics["retrieved_passage_ids"],
        "semantic_passages": semantic_metrics["retrieved_passage_ids"],
    })

retrieval_detail_comparison = pd.DataFrame(comparison_details)

display(retrieval_detail_comparison)

,question_id,persona,question,bm25_hit,semantic_hit,bm25_recall,semantic_recall,bm25_ndcg,semantic_ndcg,bm25_passages,semantic_passages
0,C01Q01,prospective student,Can I study a Bachelor of Business online?,1,1,1.0,1.0,0.963940,0.796708,"[P01, P16, P15]","[P15, P01, P16]"
1,C01Q02,current student,Is there an online version of the Business degree?,1,1,0.5,1.0,0.826235,0.796708,"[P01, P16, P44]","[P15, P01, P16]"
2,C02Q01,parent/guardian,What career outcomes does the Marketing major lead to?,1,1,1.0,1.0,1.000000,1.000000,"[P02, P32, P14]","[P02, P06, P05]"
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,1,1,1.0,1.0,1.000000,1.000000,"[P02, P04, P44]","[P02, P06, P13]"
4,C03Q01,current student,Am I eligible to take a third-year elective as a first-year student?,1,1,1.0,1.0,1.000000,1.000000,"[P03, P19, P12]","[P03, P13, P32]"
5,C03Q02,current student,Can I enrol in advanced electives early?,1,1,1.0,1.0,1.000000,1.000000,"[P03, P09, P11]","[P03, P35, P13]"
6,C04Q01,parent/guardian,Does the Bachelor of Commerce prepare students for further postgraduate study?,0,0,0.0,0.0,0.000000,0.000000,"[P44, P20, P38]","[P44, P13, P12]"
7,C04Q02,current student,Can I go on to a Master's after this degree?,1,1,1.0,1.0,1.000000,0.630930,"[P04, P44, P10]","[P44, P04, P28]"
8,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?,0,1,0.0,1.0,0.000000,0.630930,"[P40, P06, P41]","[P40, P05, P06]"
9,C05Q02,prospective student,What study areas does Bachelor of Graphic Design offer?,0,1,0.0,1.0,0.000000,0.630930,"[P20, P32, P40]","[P40, P05, P24]"


In [24]:
fixed_by_semantic = retrieval_detail_comparison[
    (retrieval_detail_comparison["bm25_hit"] == 0) &
    (retrieval_detail_comparison["semantic_hit"] == 1)
]

display(fixed_by_semantic)

,question_id,persona,question,bm25_hit,semantic_hit,bm25_recall,semantic_recall,bm25_ndcg,semantic_ndcg,bm25_passages,semantic_passages
8,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?,0,1,0.0,1.0,0.0,0.63093,"[P40, P06, P41]","[P40, P05, P06]"
9,C05Q02,prospective student,What study areas does Bachelor of Graphic Design offer?,0,1,0.0,1.0,0.0,0.63093,"[P20, P32, P40]","[P40, P05, P24]"
10,C06Q01,prospective student,What career opportunities are available after completing the Bachelor of Graphic Design?,0,1,0.0,1.0,0.0,0.63093,"[P44, P23, P40]","[P41, P06, P40]"
32,C23Q01,parent/guardian,Will students get any industry experience while completing Commerce?,0,1,0.0,1.0,0.0,0.63093,"[P33, P25, P20]","[P12, P13, P04]"
33,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,0,1,0.0,1.0,0.0,1.00000,"[P14, P32, P44]","[P39, P09, P15]"


In [25]:
semantic_regressions = retrieval_detail_comparison[
    (retrieval_detail_comparison["bm25_hit"] == 1) &
    (retrieval_detail_comparison["semantic_hit"] == 0)
]

display(semantic_regressions)

,question_id,persona,question,bm25_hit,semantic_hit,bm25_recall,semantic_recall,bm25_ndcg,semantic_ndcg,bm25_passages,semantic_passages
14,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,1,0,0.5,0.0,0.137706,0.0,"[P18, P03, P17]","[P42, P20, P43]"


In [26]:
semantic_failures = retrieval_detail_comparison[
    retrieval_detail_comparison["semantic_hit"] == 0
]

display(semantic_failures)

,question_id,persona,question,bm25_hit,semantic_hit,bm25_recall,semantic_recall,bm25_ndcg,semantic_ndcg,bm25_passages,semantic_passages
6,C04Q01,parent/guardian,Does the Bachelor of Commerce prepare students for further postgraduate study?,0,0,0.0,0.0,0.000000,0.0,"[P44, P20, P38]","[P44, P13, P12]"
12,C07Q01,prospective student,What does the Bachelor of Games focus on?,0,0,0.0,0.0,0.000000,0.0,"[P24, P32, P39]","[P42, P20, P19]"
13,C07Q02,current student,What areas will I study in the Bachelor of Games?,0,0,0.0,0.0,0.000000,0.0,"[P20, P35, P43]","[P20, P42, P38]"
14,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,1,0,0.5,0.0,0.137706,0.0,"[P18, P03, P17]","[P42, P20, P43]"
23,C14Q01,prospective student,What software should I be expected to learn during the Bachelor of Games?,0,0,0.0,0.0,0.000000,0.0,"[P16, P17, P44]","[P42, P38, P20]"
49,C50Q01,parent/guardian,What careers can my child pursue after completing the Bachelor of Games?,0,0,0.0,0.0,0.000000,0.0,"[P44, P41, P06]","[P20, P38, P42]"


In [27]:
persona_comparison = (
    retrieval_detail_comparison
    .groupby("persona")
    .agg(
        bm25_hit=("bm25_hit", "mean"),
        semantic_hit=("semantic_hit", "mean"),

        bm25_recall=("bm25_recall", "mean"),
        semantic_recall=("semantic_recall", "mean"),

        bm25_ndcg=("bm25_ndcg", "mean"),
        semantic_ndcg=("semantic_ndcg", "mean")
    )
    .reset_index()
)

display(persona_comparison)

,persona,bm25_hit,semantic_hit,bm25_recall,semantic_recall,bm25_ndcg,semantic_ndcg
0,current student,0.947368,0.947368,0.868421,0.894737,0.850194,0.831113
1,parent/guardian,0.666667,0.833333,0.625000,0.791667,0.596019,0.754336
2,prospective student,0.750000,0.850000,0.675000,0.825000,0.635903,0.749722


# Generate answers for DeepEval

In [28]:
ABSTENTION_TEXT = (
    "I don't have enough information in the provided RMIT sources "
    "to answer this question."
)

def build_prompt(question, context_passages):
    context = "\n\n".join(context_passages)

    return f"""You are an RMIT university information assistant.

Answer the user's question using ONLY the RMIT information provided below.

Follow these rules carefully:

1. Read ALL provided passages before answering.

2. If a passage directly answers the question, use that information.
   Do NOT abstain when the answer is explicitly stated.

3. Prioritise the passage that most specifically matches the user's question.
   For example:
   - for part-time study, prioritise information specifically about part-time study;
   - for changing majors, prioritise information specifically about changing majors;
   - for a named program, only use information that applies to that program.

4. Preserve explicit statements exactly.
   Pay particular attention to:
   "can", "cannot", "must", "must not",
   "required", "not required",
   "eligible", "not eligible",
   "exempt", "not exempt",
   "full-time", and "part-time".

   Also preserve numerical information such as years, subjects, fees,
   scores, study loads, and durations.

5. Do not infer permission, eligibility, requirements, policies, predictions,
   comparisons, or outcomes from indirect or missing information.

   The absence of information does NOT mean the answer is "no".

6. ANSWER IN A COMPLETE SENTENCE
   Give a direct, self-contained answer to the user's question.

   Do not respond with only "Yes" or "No".

   For yes-or-no questions, clearly state whether the answer is yes or no,
   then include the specific information from the RMIT sources that supports it.

   For questions beginning with "what", "how", "which", "when", or similar
   question words, answer the requested information directly.
   Do not begin these answers with "Yes" or "No".

   Make sure the answer does not contradict the evidence.

7. Before responding, check that your answer does not contradict any explicit
   statement in the provided information.

8. Only abstain if none of the provided passages contain enough information
   to answer the question.

   If there is not enough information, respond exactly with:
   "{ABSTENTION_TEXT}"

9. ANSWER DIRECTLY AND CONCISELY
   Answer in one or two complete sentences where possible.

   Include enough information to fully answer the question, but do not add
   unnecessary details.

   Do not explain your reasoning process or describe how you found the answer.
   Do not use phrases such as:
   "Based on the information provided",
   "According to the provided information",
   or "To determine this".

10. Do not invent, assume, predict, calculate, or add information that is not
    supported by the RMIT information below.

Question:
{question}

RMIT information:
{context}

Answer:
"""

def generate_answer(question, model=GENERATOR_MODEL, k=TOP_K):
    retrieved = retrieve_semantic(question, k=k)
    context_passages = [item["passage"] for item in retrieved]

    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": build_prompt(question, context_passages)
        }],
        options={
            "temperature": 0.0,
            "num_ctx": 4096
        }
    )

    return {
        "answer": response["message"]["content"].strip(),
        "retrieval_context": context_passages,
        "retrieved_passage_ids": [item["passage_id"] for item in retrieved]
    }


## Build a small stratified sample

In [29]:
qrel_question_ids = set(qrels["question_id"])

deepeval_pool = topics[
    (topics["status"] == "known") &
    (topics["question_id"].isin(qrel_question_ids))
].copy()

print("Eligible questions per persona:")
display(deepeval_pool["persona"].value_counts())

deepeval_sample = (
    deepeval_pool
    .groupby("persona", group_keys=False)
    .sample(
        n=DEEPEVAL_PER_PERSONA,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

display(
    deepeval_sample[
        ["question_id", "persona", "question"]
    ].sort_values(["persona", "question_id"])
)

print("DeepEval sample size:", len(deepeval_sample))


Eligible questions per persona:


persona
prospective student    20
current student        19
parent/guardian        12
Name: count, dtype: int64

,question_id,persona,question
0,C01Q02,current student,Is there an online version of the Business degree?
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?
5,C26Q01,parent/guardian,What student supports are available for my child?
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?
10,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?


DeepEval sample size: 12


## Generate answers first

In [30]:
generation_rows = []

for i, row in deepeval_sample.iterrows():
    print(
        f"[{i + 1}/{len(deepeval_sample)}] "
        f"Generating {row['question_id']}..."
    )

    result = generate_answer(row["question"])

    generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"]
    })

    pd.DataFrame(generation_rows).to_pickle(
        "generation_results_progress.pkl"
    )

generation_results = pd.DataFrame(generation_rows)

display(
    generation_results[
        ["question_id", "persona", "question", "answer", "retrieved_passage_ids"]
    ]
)

print("Generation complete.")


[1/12] Generating C01Q02...
[2/12] Generating C06Q02...
[3/12] Generating C20Q01...
[4/12] Generating C02Q02...
[5/12] Generating C51Q01...
[6/12] Generating C26Q01...
[7/12] Generating C24Q01...
[8/12] Generating C17Q02...
[9/12] Generating C08Q01...
[10/12] Generating C42Q01...
[11/12] Generating C05Q01...
[12/12] Generating C09Q01...


,question_id,persona,question,answer,retrieved_passage_ids
0,C01Q02,current student,Is there an online version of the Business degree?,"There is an online version of the Business degree, specifically the full-time, online Bachelor of Business, which offers flexible online...","[P15, P01, P16]"
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?,"Graduates of the Bachelor of Graphic Design can pursue careers such as graphic designer, UX/UI designer, brand designer, illustrator, pu...","[P41, P06, P40]"
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can consider completing four courses per year over six years, as part-ti...","[P15, P35, P27]"
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"Students majoring in Marketing can expect roles in digital marketing, brand management, campaign strategy, and customer analytics across...","[P02, P06, P13]"
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?,"RMIT's Bachelor of Graphic Design and Game Design programs provide students with opportunities to work on real-world design projects, in...","[P41, P24, P23]"
5,C26Q01,parent/guardian,What student supports are available for my child?,"RMIT students have access to various support services, including Student Connect for administration and academic advice, IT Service Conn...","[P45, P33, P30]"
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,"The Bachelor of Business costs AU$47,040 annually for international students, and domestic students may have Commonwealth Supported Plac...","[P39, P09, P15]"
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?,"A student cannot switch majors halfway through the Bachelor of Laws + Commerce program. This is because, once a student has commenced th...","[P26, P25, P27]"
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,"To apply for the Bachelor of Games, domestic students must have successfully completed the Victorian Certificate of Education (VCE) or e...","[P42, P20, P43]"
9,C42Q01,prospective student,How long does the online Bachelor of Business take if I study part-time?,"The online Bachelor of Business takes 6 years to complete if you study part-time, as part-time students may complete 4 courses per year ...","[P15, P16, P01]"


Generation complete.


## Readability Evaluation of Generated Answers

Readability is evaluated on the generated answers from the 12-question stratified sample. The metrics describe the linguistic complexity of each response and can be compared across personas.


In [31]:
def count_syllables(word):
    """Approximate the number of syllables in an English word."""
    word = re.sub(r"[^a-zA-Z]", "", str(word)).lower()

    if not word:
        return 0

    # Treat a final silent 'e' as non-syllabic in most cases.
    word = re.sub(r"e$", "", word)
    vowel_groups = re.findall(r"[aeiouy]+", word)
    syllables = len(vowel_groups)

    return max(1, syllables)


def readability_metrics(text):
    """Calculate descriptive readability metrics for a generated answer."""
    text = str(text).strip()
    words = re.findall(r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b", text)
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

    word_count = len(words)
    sentence_count = len(sentences)
    syllable_count = sum(count_syllables(word) for word in words)

    if word_count == 0 or sentence_count == 0:
        return {
            "word_count": 0,
            "sentence_count": 0,
            "complex_word_count": 0,
            "avg_words_per_sentence": np.nan,
            "avg_syllables_per_word": np.nan,
            "flesch_reading_ease": np.nan,
            "flesch_kincaid_grade": np.nan,
            "gunning_fog": np.nan,
            "smog_index": np.nan,
        }

    words_per_sentence = word_count / sentence_count
    syllables_per_word = syllable_count / word_count

    # Complex words are approximated as words containing 3+ syllables.
    complex_word_count = sum(count_syllables(word) >= 3 for word in words)

    # Standard Flesch formulas.
    flesch_reading_ease = (
        206.835
        - 1.015 * words_per_sentence
        - 84.6 * syllables_per_word
    )

    flesch_kincaid_grade = (
        0.39 * words_per_sentence
        + 11.8 * syllables_per_word
        - 15.59
    )

    # Gunning Fog Index.
    gunning_fog = 0.4 * (
        words_per_sentence
        + 100 * (complex_word_count / word_count)
    )

    # SMOG Index.
    # The standard formula is most appropriate for longer passages;
    # short chatbot answers should therefore be interpreted cautiously.
    smog_index = (
        1.0430 * np.sqrt(complex_word_count * (30 / sentence_count))
        + 3.1291
    )

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "complex_word_count": complex_word_count,
        "avg_words_per_sentence": words_per_sentence,
        "avg_syllables_per_word": syllables_per_word,
        "flesch_reading_ease": flesch_reading_ease,
        "flesch_kincaid_grade": flesch_kincaid_grade,
        "gunning_fog": gunning_fog,
        "smog_index": smog_index,
    }


In [32]:
readability_rows = []

for _, row in generation_results.iterrows():
    metrics = readability_metrics(row["answer"])

    readability_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        **metrics
    })

readability_results = pd.DataFrame(readability_rows)

display(
    readability_results[
        [
            "question_id",
            "persona",
            "word_count",
            "sentence_count",
            "avg_words_per_sentence",
            "avg_syllables_per_word",
            "complex_word_count",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ].round(2)
)


,question_id,persona,word_count,sentence_count,avg_words_per_sentence,avg_syllables_per_word,complex_word_count,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
0,C01Q02,current student,27,1,27.0,1.81,4,25.90,16.35,16.73,14.55
1,C06Q02,current student,32,1,32.0,2.00,10,5.16,20.49,25.30,21.19
2,C20Q01,current student,29,1,29.0,1.62,4,40.29,14.84,17.12,14.55
3,C02Q02,current student,36,1,36.0,2.22,15,-17.70,24.67,31.07,25.25
4,C51Q01,parent/guardian,36,1,36.0,2.00,7,1.10,22.05,22.18,18.24
5,C26Q01,parent/guardian,31,1,31.0,1.94,6,11.63,19.34,20.14,17.12
6,C24Q01,parent/guardian,24,1,24.0,1.92,8,20.33,16.39,22.93,19.29
7,C17Q02,parent/guardian,35,2,17.5,1.60,5,53.71,10.12,12.71,12.16
8,C08Q01,prospective student,48,1,48.0,1.71,10,13.59,23.29,27.53,21.19
9,C42Q01,prospective student,23,1,23.0,1.61,2,47.39,12.36,12.68,11.21


In [33]:
readability_summary = pd.DataFrame({
    "metric": [
        "Average words per sentence",
        "Average syllables per word",
        "Average complex words per answer",
        "Flesch Reading Ease",
        "Flesch-Kincaid Grade Level",
        "Gunning Fog Index",
        "SMOG Index"
    ],
    "score": [
        readability_results["avg_words_per_sentence"].mean(),
        readability_results["avg_syllables_per_word"].mean(),
        readability_results["complex_word_count"].mean(),
        readability_results["flesch_reading_ease"].mean(),
        readability_results["flesch_kincaid_grade"].mean(),
        readability_results["gunning_fog"].mean(),
        readability_results["smog_index"].mean()
    ]
})

print("Overall readability metrics:")
display(readability_summary.round(2))


Overall readability metrics:


,metric,score
0,Average words per sentence,30.21
1,Average syllables per word,1.85
2,Average complex words per answer,7.25
3,Flesch Reading Ease,19.41
4,Flesch-Kincaid Grade Level,18.06
5,Gunning Fog Index,21.05
6,SMOG Index,17.69


In [34]:
readability_by_persona = (
    readability_results
    .groupby("persona")[
        [
            "avg_words_per_sentence",
            "avg_syllables_per_word",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ]
    .mean()
    .round(2)
)

print("Readability by persona:")
display(readability_by_persona)


Readability by persona:


,avg_words_per_sentence,avg_syllables_per_word,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
persona,,,,,,
current student,31.00,1.91,13.41,19.09,22.55,18.89
parent/guardian,27.12,1.86,21.69,16.97,19.49,16.70
prospective student,32.50,1.78,23.13,18.11,21.10,17.47


In [35]:
readability_results.to_csv("readability_results.csv", index=False)
readability_summary.to_csv("readability_summary.csv", index=False)
readability_by_persona.to_csv("readability_by_persona.csv")

print("Readability results saved.")


Readability results saved.


# DeepEval

In [36]:
from deepeval.models import OllamaModel
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

judge_model = OllamaModel(
    model=JUDGE_MODEL,   
    base_url="http://localhost:11434",
    temperature=0
)

answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=False,
    async_mode=False
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=False,
    async_mode=False
)

print("DeepEval judge:", JUDGE_MODEL)

DeepEval judge: qwen3:1.7b


## Answer Relevancy

In [37]:
relevancy_rows = []

for i, row in generation_results.iterrows():
    print(
        f"[{i + 1}/{len(generation_results)}] "
        f"Answer Relevancy — {row['question_id']}"
    )

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"]
    )

    try:
        answer_relevancy_metric.measure(test_case)
        score = answer_relevancy_metric.score
        error = None

    except Exception as e:
        score = np.nan
        error = str(e)

    relevancy_rows.append({
        "question_id": row["question_id"],
        "answer_relevancy": score,
        "relevancy_error": error
    })

    pd.DataFrame(relevancy_rows).to_csv(
        "deepeval_relevancy_progress.csv",
        index=False
    )

    print(f"Score: {score}")

relevancy_results = pd.DataFrame(relevancy_rows)

display(relevancy_results)

/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[1/12] Answer Relevancy — C01Q02


/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Score: 1.0
[2/12] Answer Relevancy — C06Q02


/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

Score: 1.0
[11/12] Answer Relevancy — C05Q01


/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Score: 1.0
[12/12] Answer Relevancy — C09Q01


Score: 0.5


,question_id,answer_relevancy,relevancy_error
0,C01Q02,1.000000,None
1,C06Q02,1.000000,None
2,C20Q01,0.333333,None
3,C02Q02,0.666667,None
4,C51Q01,1.000000,None
5,C26Q01,1.000000,None
6,C24Q01,1.000000,None
7,C17Q02,1.000000,None
8,C08Q01,1.000000,None
9,C42Q01,1.000000,None


## Faithfulness

In [40]:
faithfulness_rows = []

for i, row in generation_results.iterrows():
    print(
        f"[{i + 1}/{len(generation_results)}] "
        f"Faithfulness — {row['question_id']}"
    )

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        retrieval_context=row["retrieval_context"]
    )

    try:
        faithfulness_metric.measure(test_case)
        score = faithfulness_metric.score
        error = None
    except Exception as e:
        score = np.nan
        error = str(e)

    faithfulness_rows.append({
        "question_id": row["question_id"],
        "faithfulness": score,
        "faithfulness_error": error
    })

    pd.DataFrame(faithfulness_rows).to_csv(
        "deepeval_faithfulness_progress.csv",
        index=False
    )

    print(f"Score: {score}")

faithfulness_results = pd.DataFrame(faithfulness_rows)
display(faithfulness_results)

/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[1/12] Faithfulness — C01Q02


/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Score: 1.0
[2/12] Faithfulness — C06Q02


/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

Score: 1.0
[11/12] Faithfulness — C05Q01


/opt/anaconda3/envs/course_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Score: 1.0
[12/12] Faithfulness — C09Q01


Score: 1.0


,question_id,faithfulness,faithfulness_error
0,C01Q02,1.0,None
1,C06Q02,1.0,None
2,C20Q01,1.0,None
3,C02Q02,1.0,None
4,C51Q01,1.0,None
5,C26Q01,1.0,None
6,C24Q01,1.0,None
7,C17Q02,1.0,None
8,C08Q01,1.0,None
9,C42Q01,1.0,None


In [41]:
question_ids = [
    # Current student
    "C01Q02",
    "C06Q02",
    "C02Q02",
    "C20Q01",

    # Parent/guardian
    "C51Q01",
    "C26Q01",   # replaces C25Q01
    "C24Q01",
    "C17Q02",

    # Prospective student
    "C08Q01",
    "C42Q01",
    "C05Q01",
    "C09Q01"
]

for qid in question_ids:
    row = generation_results[
        generation_results["question_id"] == qid
    ].iloc[0]

    print("\nQUESTION:", qid)
    print(row["question"])
    print("\nANSWER:")
    print(row["answer"])
    print("\nRETRIEVAL CONTEXT:")
    for passage in row["retrieval_context"]:
        print("-", passage)

    print("\n" + "=" * 80)


QUESTION: C01Q02
Is there an online version of the Business degree?

ANSWER:
There is an online version of the Business degree, specifically the full-time, online Bachelor of Business, which offers flexible online learning in small cohorts of around 25 students.

RETRIEVAL CONTEXT:
- Students studying the Bachelor of Business online can choose between full-time and part-time study. The degree contains 24 courses. Full-time students complete eight courses per year over three years, while part-time students may complete four courses per year over six years.
- The Bachelor of Business is available to study online at RMIT. The full-time, online Bachelor of Business takes 36 months to complete and offers flexible online learning in small cohorts of around 25 students with support from an Online Facilitator. The online degree is not available to international students intending to study on a student visa.
- The expected study commitment for the online Bachelor of Business is approximately 1

In [44]:
deepeval_results = relevancy_results.merge(
    faithfulness_results,
    on="question_id",
    how="inner"
)

display(deepeval_results)

,question_id,answer_relevancy,relevancy_error,faithfulness,faithfulness_error
0,C01Q02,1.000000,None,1.0,None
1,C06Q02,1.000000,None,1.0,None
2,C20Q01,0.333333,None,1.0,None
3,C02Q02,0.666667,None,1.0,None
4,C51Q01,1.000000,None,1.0,None
5,C26Q01,1.000000,None,1.0,None
6,C24Q01,1.000000,None,1.0,None
7,C17Q02,1.000000,None,1.0,None
8,C08Q01,1.000000,None,1.0,None
9,C42Q01,1.000000,None,1.0,None


In [45]:
deepeval_summary = pd.DataFrame({
    "metric": ["Answer Relevancy", "Faithfulness"],
    "score": [
        deepeval_results["answer_relevancy"].mean(),
        deepeval_results["faithfulness"].mean()
    ]
})

display(deepeval_summary.round(3))

,metric,score
0,Answer Relevancy,0.875
1,Faithfulness,1.000


In [48]:
deepeval_results = deepeval_results.merge(
    generation_results[["question_id", "persona"]],
    on="question_id",
    how="left"
)

display(deepeval_results)

,question_id,answer_relevancy,relevancy_error,faithfulness,faithfulness_error,persona
0,C01Q02,1.000000,None,1.0,None,current student
1,C06Q02,1.000000,None,1.0,None,current student
2,C20Q01,0.333333,None,1.0,None,current student
3,C02Q02,0.666667,None,1.0,None,current student
4,C51Q01,1.000000,None,1.0,None,parent/guardian
5,C26Q01,1.000000,None,1.0,None,parent/guardian
6,C24Q01,1.000000,None,1.0,None,parent/guardian
7,C17Q02,1.000000,None,1.0,None,parent/guardian
8,C08Q01,1.000000,None,1.0,None,prospective student
9,C42Q01,1.000000,None,1.0,None,prospective student


In [49]:
deepeval_by_persona = (
    deepeval_results
    .groupby("persona")[["answer_relevancy", "faithfulness"]]
    .mean()
    .round(3)
)

display(deepeval_by_persona)

,answer_relevancy,faithfulness
persona,,
current student,0.750,1.0
parent/guardian,1.000,1.0
prospective student,0.875,1.0


In [51]:
deepeval_results = deepeval_results.merge(
    generation_results[
        [
            "question_id",
            "question",
            "answer",
            "retrieved_passage_ids"
        ]
    ],
    on="question_id",
    how="left"
)

In [52]:
weak_cases = deepeval_results[
    (deepeval_results["answer_relevancy"] <= 0.5) |
    (deepeval_results["faithfulness"] <= 0.5)
].copy()

display(
    weak_cases[
        [
            "question_id",
            "persona",
            "question",
            "answer",
            "retrieved_passage_ids",
            "answer_relevancy",
            "faithfulness"
        ]
    ]
)

print("Weak cases:", len(weak_cases))

,question_id,persona,question,answer,retrieved_passage_ids,answer_relevancy,faithfulness
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can consider completing four courses per year over six years, as part-ti...","[P15, P35, P27]",0.333333,1.0
11,C09Q01,prospective student,"Do I need a background in animation or games to get into the Master of Animation, Games, and Interactivity program?","You do not need a background in animation or games to get into the Master of Animation, Games, and Interactivity program, as applicants ...","[P28, P18, P07]",0.500000,1.0


Weak cases: 2


# Robustness (Out-of-KB Refusal & False Abstention)

## Categorise the out-of-KB questions

- **predictive** — asks about the future / not-yet-decided information
- **subjective** — asks for an opinion or comparison with no objective answer ("hardest degree", "best lecturer")
- **personal_circumstance** — depends on applicant-specific info the KB can never contain (chances of acceptance)
- **out_of_scope_topic** — a real, objective topic, just outside the course-structure domain (salary, software, job market)
- **kb_gap** — a genuinely in-scope, answerable-in-principle question that the current KB simply doesn't cover

In [53]:
OOK_SUBTYPES = {
    "C14Q02": "out_of_scope_topic",       # software needed for Bachelor of Games
    "C18Q01": "kb_gap",                   # transferring Business <-> Commerce
    "C27Q01": "subjective",               # Graphic Design vs Games difficulty
    "C27Q02": "subjective",
    "C28Q01": "predictive",               # ATAR in 2028
    "C28Q02": "predictive",
    "C29Q01": "subjective",               # "best" employment rate
    "C29Q02": "out_of_scope_topic",       # employability after Games
    "C30Q01": "subjective",               # "best" lecturer
    "C31Q01": "personal_circumstance",    # chances of acceptance
    "C31Q02": "personal_circumstance",
    "C32Q01": "out_of_scope_topic",       # laptop recommendation
    "C33Q01": "out_of_scope_topic",       # salary
    "C33Q02": "out_of_scope_topic",
    "C34Q01": "out_of_scope_topic",       # job openings
    "C35Q01": "predictive",               # new majors in 2 years
    "C35Q02": "predictive",
    "C52Q01": "predictive",               # job market in 5 years
}

ook_topics = topics[topics["status"] == "out-of-kb"].copy()
ook_topics["subtype"] = ook_topics["question_id"].map(OOK_SUBTYPES)

missing = ook_topics[ook_topics["subtype"].isna()]
if len(missing):
    print("WARNING: unmapped out-of-kb question_ids, add them to OOK_SUBTYPES:")
    display(missing[["question_id", "question"]])

display(ook_topics[["question_id", "persona", "subtype", "question"]])
print("Out-of-KB questions:", len(ook_topics))
print(ook_topics["subtype"].value_counts())


,question_id,question
36,C25Q01,What computer specifications will my child need for the Bachelor of games?


,question_id,persona,subtype,question
24,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?
29,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?
36,C25Q01,parent/guardian,NaN,What computer specifications will my child need for the Bachelor of games?
38,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?
39,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?"
40,C28Q01,prospective student,predictive,Is the ATAR for Bachelor of Business going to change in 2028?
41,C28Q02,prospective student,predictive,What ATAR will I need for the Bachelor of Business in 2028?
42,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?
43,C29Q01,prospective student,subjective,Which degree has the best employment rate?
44,C30Q01,current student,subjective,Which lecturer is the best for the Bachelor of Commerce subjects?


Out-of-KB questions: 19
subtype
out_of_scope_topic       6
predictive               5
subjective               4
personal_circumstance    2
kb_gap                   1
Name: count, dtype: int64


## Generate answers for the out-of-KB questions

In [54]:
ook_generation_rows = []

for i, row in ook_topics.iterrows():
    print(f"[{len(ook_generation_rows) + 1}/{len(ook_topics)}] Generating {row['question_id']}...")

    result = generate_answer(row["question"])

    ook_generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "subtype": row["subtype"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"],
    })

    pd.DataFrame(ook_generation_rows).to_pickle("ook_generation_results_progress.pkl")

ook_generation_results = pd.DataFrame(ook_generation_rows)
display(ook_generation_results[["question_id", "persona", "subtype", "question", "answer"]])
print("Out-of-KB generation complete.")


[1/19] Generating C14Q02...
[2/19] Generating C18Q01...
[3/19] Generating C25Q01...
[4/19] Generating C27Q01...
[5/19] Generating C27Q02...
[6/19] Generating C28Q01...
[7/19] Generating C28Q02...
[8/19] Generating C29Q02...
[9/19] Generating C29Q01...
[10/19] Generating C30Q01...
[11/19] Generating C31Q01...
[12/19] Generating C31Q02...
[13/19] Generating C32Q01...
[14/19] Generating C33Q01...
[15/19] Generating C33Q02...
[16/19] Generating C34Q01...
[17/19] Generating C35Q01...
[18/19] Generating C35Q02...
[19/19] Generating C52Q01...


,question_id,persona,subtype,question,answer
0,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?,"To complete the Bachelor of Games, you do not need to have specific software installed, as the degree focuses on broad capabilities acro..."
1,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?,"You can transfer between the Bachelor of Business and the Bachelor of Commerce, but only after completing at least 64 credit points of s..."
2,C25Q01,parent/guardian,NaN,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.
3,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?,"The Bachelor of Graphic Design is not necessarily harder than the Bachelor of Games, as both programs have different focuses and emphasi..."
4,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?","The Bachelor of Games is more difficult than the Bachelor of Graphic Design, as it provides specialist training in computer-generated de..."
5,C28Q01,prospective student,predictive,Is the ATAR for Bachelor of Business going to change in 2028?,I don't have enough information in the provided RMIT sources to answer this question.
6,C28Q02,prospective student,predictive,What ATAR will I need for the Bachelor of Business in 2028?,I don't have enough information in the provided RMIT sources to answer this question.
7,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?,"After completing the Bachelor of Games, you can expect to have a strong employability profile, with a professional portfolio that showca..."
8,C29Q01,prospective student,subjective,Which degree has the best employment rate?,"The Marketing major has the best employment rate, with entry-level salaries starting around AU$70,000 and rising to AU$100,000+ for seni..."
9,C30Q01,current student,subjective,Which lecturer is the best for the Bachelor of Commerce subjects?,I don't have enough information in the provided RMIT sources to answer this question.


Out-of-KB generation complete.


## Classify each answer as abstained or attempted

In [55]:
import difflib

def fuzzy_is_abstention(answer, threshold=0.6):
    """Cheap heuristic: does the answer closely resemble the canonical abstention text,
    or open with an equivalent refusal phrase?"""
    answer_clean = str(answer).strip().lower()
    ratio = difflib.SequenceMatcher(
        None, answer_clean, ABSTENTION_TEXT.lower()
    ).ratio()

    refusal_phrases = [
        "don't have enough information",
        "do not have enough information",
        "not enough information",
        "cannot answer",
        "can't answer",
        "unable to answer",
        "no information",
        "not provided",
        "not mentioned in the",
    ]
    phrase_hit = any(p in answer_clean for p in refusal_phrases)

    return (ratio >= threshold) or phrase_hit


JUDGE_ABSTENTION_PROMPT = """You are auditing an RMIT university chatbot.

Question asked:
{question}

Chatbot's answer:
{answer}

Decide whether the chatbot actually ANSWERED the question with specific information,
or whether it ABSTAINED / refused / said it did not have enough information
(including partial refusals, hedged non-answers, or "I'm not sure" type responses).

Respond with exactly one word: ANSWERED or ABSTAINED."""


def judge_is_abstention(question, answer, model=JUDGE_MODEL, retries=2):
    prompt = JUDGE_ABSTENTION_PROMPT.format(question=question, answer=answer)

    for attempt in range(retries):
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.0}
        )
        verdict = response["message"]["content"].strip().upper()

        if "ABSTAIN" in verdict:
            return True
        if "ANSWER" in verdict:
            return False

    return fuzzy_is_abstention(answer)


def classify_abstention_row(question, answer):
    fuzzy = fuzzy_is_abstention(answer)
    judged = judge_is_abstention(question, answer)

    return {
        "fuzzy_abstained": fuzzy,
        "judge_abstained": judged,
        "methods_agree": fuzzy == judged,

        "is_abstained": judged,
    }


In [56]:
classification_rows = []

for i, row in ook_generation_results.iterrows():
    print(f"[{i + 1}/{len(ook_generation_results)}] Classifying {row['question_id']}...")
    result = classify_abstention_row(row["question"], row["answer"])
    classification_rows.append({"question_id": row["question_id"], **result})

ook_classified = ook_generation_results.merge(
    pd.DataFrame(classification_rows), on="question_id", how="left"
)

disagreements = ook_classified[~ook_classified["methods_agree"]]
if len(disagreements):
    print(f"{len(disagreements)} question(s) where the fuzzy check and judge disagreed — worth a manual read:")
    display(disagreements[["question_id", "question", "answer", "fuzzy_abstained", "judge_abstained"]])

display(ook_classified[["question_id", "persona", "subtype", "question", "answer", "is_abstained"]])


[1/19] Classifying C14Q02...
[2/19] Classifying C18Q01...
[3/19] Classifying C25Q01...
[4/19] Classifying C27Q01...
[5/19] Classifying C27Q02...
[6/19] Classifying C28Q01...
[7/19] Classifying C28Q02...
[8/19] Classifying C29Q02...
[9/19] Classifying C29Q01...
[10/19] Classifying C30Q01...
[11/19] Classifying C31Q01...
[12/19] Classifying C31Q02...
[13/19] Classifying C32Q01...
[14/19] Classifying C33Q01...
[15/19] Classifying C33Q02...
[16/19] Classifying C34Q01...
[17/19] Classifying C35Q01...
[18/19] Classifying C35Q02...
[19/19] Classifying C52Q01...
3 question(s) where the fuzzy check and judge disagreed — worth a manual read:


,question_id,question,answer,fuzzy_abstained,judge_abstained
0,C14Q02,What software do I need to have in order to complete the Bachelor of games?,"To complete the Bachelor of Games, you do not need to have specific software installed, as the degree focuses on broad capabilities acro...",False,True
8,C29Q01,Which degree has the best employment rate?,"The Marketing major has the best employment rate, with entry-level salaries starting around AU$70,000 and rising to AU$100,000+ for seni...",False,True
15,C34Q01,What job openings exist in the industry for the Bachelor of Games?,"The Bachelor of Games offers students job openings in the industry through its internship and research-focused study opportunities, as w...",False,True


,question_id,persona,subtype,question,answer,is_abstained
0,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?,"To complete the Bachelor of Games, you do not need to have specific software installed, as the degree focuses on broad capabilities acro...",True
1,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?,"You can transfer between the Bachelor of Business and the Bachelor of Commerce, but only after completing at least 64 credit points of s...",False
2,C25Q01,parent/guardian,NaN,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.,True
3,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?,"The Bachelor of Graphic Design is not necessarily harder than the Bachelor of Games, as both programs have different focuses and emphasi...",False
4,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?","The Bachelor of Games is more difficult than the Bachelor of Graphic Design, as it provides specialist training in computer-generated de...",False
5,C28Q01,prospective student,predictive,Is the ATAR for Bachelor of Business going to change in 2028?,I don't have enough information in the provided RMIT sources to answer this question.,True
6,C28Q02,prospective student,predictive,What ATAR will I need for the Bachelor of Business in 2028?,I don't have enough information in the provided RMIT sources to answer this question.,True
7,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?,"After completing the Bachelor of Games, you can expect to have a strong employability profile, with a professional portfolio that showca...",False
8,C29Q01,prospective student,subjective,Which degree has the best employment rate?,"The Marketing major has the best employment rate, with entry-level salaries starting around AU$70,000 and rising to AU$100,000+ for seni...",True
9,C30Q01,current student,subjective,Which lecturer is the best for the Bachelor of Commerce subjects?,I don't have enough information in the provided RMIT sources to answer this question.,True


## Out-of-KB Abstention Rate

In [57]:
ook_abstention_rate = ook_classified["is_abstained"].mean()

print(f"Out-of-KB Abstention Rate (overall): {ook_abstention_rate:.1%}  "
      f"({ook_classified['is_abstained'].sum()}/{len(ook_classified)} correctly refused)")

ook_by_subtype = (
    ook_classified
    .groupby("subtype")["is_abstained"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "abstention_rate", "sum": "n_correct_refusals", "count": "n_questions"})
    .round(3)
    .sort_values("abstention_rate")
)
display(ook_by_subtype)

ook_by_persona = (
    ook_classified
    .groupby("persona")["is_abstained"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "abstention_rate", "sum": "n_correct_refusals", "count": "n_questions"})
    .round(3)
)
display(ook_by_persona)

hallucinations = ook_classified[~ook_classified["is_abstained"]].copy()
print(f"\nHallucinated / attempted answers on out-of-KB questions: {len(hallucinations)}")
display(hallucinations[["question_id", "persona", "subtype", "question", "answer", "retrieved_passage_ids"]])


Out-of-KB Abstention Rate (overall): 73.7%  (14/19 correctly refused)


,abstention_rate,n_correct_refusals,n_questions
subtype,,,
kb_gap,0.000,0,1
subjective,0.500,2,4
predictive,0.800,4,5
out_of_scope_topic,0.833,5,6
personal_circumstance,1.000,2,2


,abstention_rate,n_correct_refusals,n_questions
persona,,,
current student,0.750,6,8
parent/guardian,1.000,2,2
prospective student,0.667,6,9



Hallucinated / attempted answers on out-of-KB questions: 5


,question_id,persona,subtype,question,answer,retrieved_passage_ids
1,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?,"You can transfer between the Bachelor of Business and the Bachelor of Commerce, but only after completing at least 64 credit points of s...","[P26, P25, P13]"
3,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?,"The Bachelor of Graphic Design is not necessarily harder than the Bachelor of Games, as both programs have different focuses and emphasi...","[P07, P42, P40]"
4,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?","The Bachelor of Games is more difficult than the Bachelor of Graphic Design, as it provides specialist training in computer-generated de...","[P07, P42, P40]"
7,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?,"After completing the Bachelor of Games, you can expect to have a strong employability profile, with a professional portfolio that showca...","[P19, P20, P42]"
17,C35Q02,prospective student,predictive,Are there plans for any new majors for the Bachelor of Games?,There are no plans for new majors for the Bachelor of Games.,"[P42, P20, P19]"


## False Abstentions

Of the questions the KB *can* answer, how often does the system
incorrectly refuse? Testing on all 51 `known` questions

In [58]:
known_topics = topics[topics["status"] == "known"].copy()

print("Known questions being tested for false abstention:", len(known_topics))

known_generation_rows = []

for i, row in known_topics.iterrows():
    print(f"[{len(known_generation_rows) + 1}/{len(known_topics)}] Generating {row['question_id']}...")

    result = generate_answer(row["question"])

    known_generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"],
    })

    pd.DataFrame(known_generation_rows).to_pickle("known_generation_results_progress.pkl")

known_generation_results = pd.DataFrame(known_generation_rows)
print("Known-question generation complete.")


Known questions being tested for false abstention: 51
[1/51] Generating C01Q01...
[2/51] Generating C01Q02...
[3/51] Generating C02Q01...
[4/51] Generating C02Q02...
[5/51] Generating C03Q01...
[6/51] Generating C03Q02...
[7/51] Generating C04Q01...
[8/51] Generating C04Q02...
[9/51] Generating C05Q01...
[10/51] Generating C05Q02...
[11/51] Generating C06Q01...
[12/51] Generating C06Q02...
[13/51] Generating C07Q01...
[14/51] Generating C07Q02...
[15/51] Generating C08Q01...
[16/51] Generating C08Q02...
[17/51] Generating C09Q01...
[18/51] Generating C09Q02...
[19/51] Generating C10Q01...
[20/51] Generating C10Q02...
[21/51] Generating C11Q01...
[22/51] Generating C12Q01...
[23/51] Generating C13Q01...
[24/51] Generating C14Q01...
[25/51] Generating C15Q01...
[26/51] Generating C16Q01...
[27/51] Generating C17Q01...
[28/51] Generating C17Q02...
[29/51] Generating C19Q01...
[30/51] Generating C20Q01...
[31/51] Generating C21Q01...
[32/51] Generating C22Q01...
[33/51] Generating C23Q01..

In [59]:
known_classification_rows = []

for i, row in known_generation_results.iterrows():
    print(f"[{i + 1}/{len(known_generation_results)}] Classifying {row['question_id']}...")
    result = classify_abstention_row(row["question"], row["answer"])
    known_classification_rows.append({"question_id": row["question_id"], **result})

known_classified = known_generation_results.merge(
    pd.DataFrame(known_classification_rows), on="question_id", how="left"
)

false_abstention_rate = known_classified["is_abstained"].mean()

print(f"False Abstention Rate: {false_abstention_rate:.1%}  "
      f"({known_classified['is_abstained'].sum()}/{len(known_classified)} known questions wrongly refused)")

false_abstentions = known_classified[known_classified["is_abstained"]].copy()
display(false_abstentions[["question_id", "persona", "question", "answer", "retrieved_passage_ids"]])


[1/51] Classifying C01Q01...
[2/51] Classifying C01Q02...
[3/51] Classifying C02Q01...
[4/51] Classifying C02Q02...
[5/51] Classifying C03Q01...
[6/51] Classifying C03Q02...
[7/51] Classifying C04Q01...
[8/51] Classifying C04Q02...
[9/51] Classifying C05Q01...
[10/51] Classifying C05Q02...
[11/51] Classifying C06Q01...
[12/51] Classifying C06Q02...
[13/51] Classifying C07Q01...
[14/51] Classifying C07Q02...
[15/51] Classifying C08Q01...
[16/51] Classifying C08Q02...
[17/51] Classifying C09Q01...
[18/51] Classifying C09Q02...
[19/51] Classifying C10Q01...
[20/51] Classifying C10Q02...
[21/51] Classifying C11Q01...
[22/51] Classifying C12Q01...
[23/51] Classifying C13Q01...
[24/51] Classifying C14Q01...
[25/51] Classifying C15Q01...
[26/51] Classifying C16Q01...
[27/51] Classifying C17Q01...
[28/51] Classifying C17Q02...
[29/51] Classifying C19Q01...
[30/51] Classifying C20Q01...
[31/51] Classifying C21Q01...
[32/51] Classifying C22Q01...
[33/51] Classifying C23Q01...
[34/51] Classifying

,question_id,persona,question,answer,retrieved_passage_ids
17,C09Q02,prospective student,"Can I apply for the Master of Animation, Games, and Interactivity without having studied animation or games before?","You can apply for the Master of Animation, Games, and Interactivity without having studied animation or games before, as applicants with...","[P28, P18, P20]"
24,C15Q01,current student,Can I go on to a Master's after completing a Bachelor of Business?,"You can go on to a Master's after completing a Bachelor of Business, as you may be eligible for entry into the Master of Commerce after ...","[P44, P04, P10]"
29,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can consider completing four courses per year over six years, as part-ti...","[P15, P35, P27]"


### Cross-check false abstentions against retrieval quality

In [60]:
false_abstentions_with_retrieval = false_abstentions.merge(
    retrieval_results[["question_id", f"hit@{TOP_K}"]], on="question_id", how="left"
)
display(false_abstentions_with_retrieval[
    ["question_id", "persona", "question", f"hit@{TOP_K}", "answer"]
])

n_generation_fault = (false_abstentions_with_retrieval[f"hit@{TOP_K}"] == 1).sum()
print(f"\nFalse abstentions where retrieval WAS successful (generation-stage fault): {n_generation_fault}")


,question_id,persona,question,hit@3,answer
0,C09Q02,prospective student,"Can I apply for the Master of Animation, Games, and Interactivity without having studied animation or games before?",1,"You can apply for the Master of Animation, Games, and Interactivity without having studied animation or games before, as applicants with..."
1,C15Q01,current student,Can I go on to a Master's after completing a Bachelor of Business?,1,"You can go on to a Master's after completing a Bachelor of Business, as you may be eligible for entry into the Master of Commerce after ..."
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,1,"To structure your degree part-time while working part-time, you can consider completing four courses per year over six years, as part-ti..."



False abstentions where retrieval WAS successful (generation-stage fault): 3


## Robustness confusion matrix

In [61]:
confusion_rows = [
    {"kb_status": "known", "model_behaviour": "answered",
     "count": int((~known_classified["is_abstained"]).sum()), "label": "Correct answer"},
    {"kb_status": "known", "model_behaviour": "abstained",
     "count": int(known_classified["is_abstained"].sum()), "label": "False abstention"},
    {"kb_status": "out-of-kb", "model_behaviour": "abstained",
     "count": int(ook_classified["is_abstained"].sum()), "label": "Correct refusal"},
    {"kb_status": "out-of-kb", "model_behaviour": "answered",
     "count": int((~ook_classified["is_abstained"]).sum()), "label": "Hallucination"},
]

confusion_df = pd.DataFrame(confusion_rows)
display(confusion_df)

robustness_summary = pd.DataFrame({
    "metric": [
        "Out-of-KB Abstention Rate (correct refusals)",
        "Hallucination Rate on out-of-KB questions",
        "False Abstention Rate (on known questions)",
        "Overall Robustness Accuracy",
    ],
    "score": [
        ook_classified["is_abstained"].mean(),
        (~ook_classified["is_abstained"]).mean(),
        known_classified["is_abstained"].mean(),
        (
            (~known_classified["is_abstained"]).sum() + ook_classified["is_abstained"].sum()
        ) / (len(known_classified) + len(ook_classified)),
    ],
}).round(3)

display(robustness_summary)


,kb_status,model_behaviour,count,label
0,known,answered,48,Correct answer
1,known,abstained,3,False abstention
2,out-of-kb,abstained,14,Correct refusal
3,out-of-kb,answered,5,Hallucination


,metric,score
0,Out-of-KB Abstention Rate (correct refusals),0.737
1,Hallucination Rate on out-of-KB questions,0.263
2,False Abstention Rate (on known questions),0.059
3,Overall Robustness Accuracy,0.886


## Save robustness results

In [62]:
ook_classified.to_csv("robustness_out_of_kb_results.csv", index=False)
known_classified.to_csv("robustness_false_abstention_results.csv", index=False)
confusion_df.to_csv("robustness_confusion_matrix.csv", index=False)
robustness_summary.to_csv("robustness_summary.csv", index=False)
ook_by_subtype.to_csv("robustness_by_subtype.csv")
ook_by_persona.to_csv("robustness_by_persona.csv")

print("Saved: robustness_out_of_kb_results.csv, robustness_false_abstention_results.csv, "
      "robustness_confusion_matrix.csv, robustness_summary.csv, "
      "robustness_by_subtype.csv, robustness_by_persona.csv")


Saved: robustness_out_of_kb_results.csv, robustness_false_abstention_results.csv, robustness_confusion_matrix.csv, robustness_summary.csv, robustness_by_subtype.csv, robustness_by_persona.csv


## Combine results

In [63]:
combined_results = (
    generation_results
    .merge(relevancy_results, on="question_id", how="left")
    .merge(faithfulness_results, on="question_id", how="left")
    .merge(readability_results, on=["question_id", "persona"], how="left")
)

display(
    combined_results[
        [
            "question_id",
            "persona",
            "question",
            "answer",
            "answer_relevancy",
            "faithfulness",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ]
)


,question_id,persona,question,answer,answer_relevancy,faithfulness,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
0,C01Q02,current student,Is there an online version of the Business degree?,"There is an online version of the Business degree, specifically the full-time, online Bachelor of Business, which offers flexible online...",1.000000,1.0,25.896667,16.354815,16.725926,14.554593
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?,"Graduates of the Bachelor of Graphic Design can pursue careers such as graphic designer, UX/UI designer, brand designer, illustrator, pu...",1.000000,1.0,5.155000,20.490000,25.300000,21.194390
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can consider completing four courses per year over six years, as part-ti...",0.333333,1.0,40.289655,14.844138,17.117241,14.554593
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"Students majoring in Marketing can expect roles in digital marketing, brand management, campaign strategy, and customer analytics across...",0.666667,1.0,-17.705000,24.672222,31.066667,25.254471
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?,"RMIT's Bachelor of Graphic Design and Game Design programs provide students with opportunities to work on real-world design projects, in...",1.000000,1.0,1.095000,22.050000,22.177778,18.243606
5,C26Q01,parent/guardian,What student supports are available for my child?,"RMIT students have access to various support services, including Student Connect for administration and academic advice, IT Service Conn...",1.000000,1.0,11.628065,19.338710,20.141935,17.122413
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,"The Bachelor of Business costs AU$47,040 annually for international students, and domestic students may have Commonwealth Supported Plac...",1.000000,1.0,20.325000,16.386667,22.933333,19.287187
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?,"A student cannot switch majors halfway through the Bachelor of Laws + Commerce program. This is because, once a student has commenced th...",1.000000,1.0,53.712500,10.115000,12.714286,12.161745
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,"To apply for the Bachelor of Games, domestic students must have successfully completed the Victorian Certificate of Education (VCE) or e...",1.000000,1.0,13.590000,23.288333,27.533333,21.194390
9,C42Q01,prospective student,How long does the online Bachelor of Business take if I study part-time?,"The online Bachelor of Business takes 6 years to complete if you study part-time, as part-time students may complete 4 courses per year ...",1.000000,1.0,47.394348,12.362609,12.678261,11.208143


In [64]:
export_df = combined_results.copy()

export_df["retrieved_passage_ids"] = export_df[
    "retrieved_passage_ids"
].apply(lambda x: " | ".join(map(str, x)))

export_df["retrieval_context"] = export_df[
    "retrieval_context"
].apply(lambda x: " ||| ".join(map(str, x)))

export_df.to_csv("combined_results.csv", index=False)
deepeval_summary.to_csv("deepeval_summary.csv", index=False)
deepeval_by_persona.to_csv("deepeval_by_persona.csv")

print("Results saved.")

Results saved.


## Overall summary table

One table combining all four evaluation dimensions (retrieval, generation, robustness, readability), for quick reference in the report and slides.

In [65]:
overall_summary = pd.concat([
    retrieval_summary.assign(section="Retrieval"),
    deepeval_summary.assign(section="Generation (DeepEval)"),
    robustness_summary.assign(section="Robustness"),
], ignore_index=True)[["section", "metric", "score"]]

display(overall_summary)
overall_summary.to_csv("overall_evaluation_summary.csv", index=False)
print("Saved: overall_evaluation_summary.csv")


,section,metric,score
0,Retrieval,Hit@3,0.803922
1,Retrieval,Recall@3,0.735294
2,Retrieval,NDCG@3,0.706352
3,Generation (DeepEval),Answer Relevancy,0.875000
4,Generation (DeepEval),Faithfulness,1.000000
5,Robustness,Out-of-KB Abstention Rate (correct refusals),0.737000
6,Robustness,Hallucination Rate on out-of-KB questions,0.263000
7,Robustness,False Abstention Rate (on known questions),0.059000
8,Robustness,Overall Robustness Accuracy,0.886000


Saved: overall_evaluation_summary.csv


## Preliminary Evaluation Summary

The preliminary evaluation indicates that the RAG system performs reasonably well overall, while also identifying areas for improvement in retrieval, answer generation, and robustness.

### Retrieval Performance

Retrieval was evaluated across 51 questions using Hit@3, Recall@3, and NDCG@3. The system achieved:

- **Hit@3 = 0.804**
- **Recall@3 = 0.735**
- **NDCG@3 = 0.706**

The Hit@3 score indicates that at least one relevant passage was retrieved within the top three results for approximately 80.4% of evaluated questions. The lower Recall@3 score suggests that, although the system frequently retrieves relevant information, it does not always retrieve all relevant passages within the top three results. The NDCG@3 score of 0.706 further indicates that relevant passages are not always ranked in the optimal order.

Performance also varied across personas. Current-student questions achieved the strongest retrieval performance (Hit@3 = 0.947, Recall@3 = 0.868, NDCG@3 = 0.850), followed by prospective-student questions (Hit@3 = 0.750, Recall@3 = 0.675, NDCG@3 = 0.636). Parent/guardian questions achieved Hit@3 = 0.667, Recall@3 = 0.625, and NDCG@3 = 0.596. These results indicated that retrieval could be improved particularly for parent/guardian and prospective-student questions.

### Retrieval Refinement: Semantic and Hybrid Retrieval

Based on the retrieval-stage error analysis, two alternative retrieval approaches were evaluated against the same 51 answerable questions: semantic retrieval using embeddings and a hybrid approach combining BM25 and semantic retrieval through Reciprocal Rank Fusion (RRF).

**Semantic Retrieval**

- **Hit@3 = 0.882**
- **Recall@3 = 0.843**
- **NDCG@3 = 0.781**

**Hybrid Retrieval**

- **Hit@3 = 0.843**
- **Recall@3 = 0.784**
- **NDCG@3 = 0.767**

Semantic retrieval achieved the strongest performance across all three metrics. Compared with the corrected BM25 baseline, Hit@3 increased from 0.804 to 0.882, Recall@3 from 0.735 to 0.843, and NDCG@3 from 0.706 to 0.781. This increased the number of questions for which at least one relevant passage was retrieved in the top three from 41/51 to 45/51.

Further inspection showed that semantic retrieval corrected five BM25 Hit@3 failures, including questions relating to Graphic Design study areas and careers, Commerce industry experience, and Bachelor of Business fees. However, it also introduced one new Hit@3 failure for a Bachelor of Games prerequisites question. Six questions therefore remained unsuccessful under semantic retrieval, with five relating to the Bachelor of Games.

The hybrid approach improved on the BM25 baseline but did not outperform semantic retrieval. As semantic retrieval produced the strongest overall retrieval performance, it was selected for the refined RAG pipeline and subsequent end-to-end evaluation.

### Refined Retrieval Performance by Persona

Semantic retrieval produced the largest improvements for the parent/guardian and prospective-student personas.

**Current Student**
- **Hit@3 = 0.947**
- **Recall@3 = 0.895**
- **NDCG@3 = 0.831**

**Parent/Guardian**
- **Hit@3 = 0.833**
- **Recall@3 = 0.792**
- **NDCG@3 = 0.754**

**Prospective Student**
- **Hit@3 = 0.850**
- **Recall@3 = 0.825**
- **NDCG@3 = 0.750**

The largest improvement occurred for parent/guardian questions, where Hit@3 increased from 0.667 to 0.833, Recall@3 from 0.625 to 0.792, and NDCG@3 from 0.596 to 0.754. Prospective-student questions also improved across all three metrics.

For current-student questions, Hit@3 remained unchanged at 0.947 and Recall@3 increased from 0.868 to 0.895. However, NDCG@3 decreased slightly from 0.850 to 0.831, indicating that semantic retrieval retrieved slightly more relevant information but did not always rank the relevant passages as highly as BM25.

Overall, semantic retrieval particularly improved performance for the parent/guardian and prospective-student personas, which showed weaker retrieval performance in the BM25 baseline.

### Generation Performance

DeepEval was applied to a stratified sample of 12 questions, with four questions selected from each persona. The preliminary system achieved:

- **Answer Relevancy = 0.861**
- **Faithfulness = 0.972**

The high Answer Relevancy score suggests that generated responses generally addressed the user's question directly. Faithfulness was also high, indicating that most answers were supported by the retrieved RMIT passages.

However, individual failures demonstrate why retrieval and generation need to be evaluated separately. For example, for **C42Q01**, the relevant passage was retrieved and stated that part-time Bachelor of Business students may complete the degree over six years. Despite this, the generated response incorrectly stated that part-time study takes 36 months. This represents a generation/faithfulness failure despite successful retrieval.

DeepEval also assigned a faithfulness score of 0 to **C25Q01**, where the system abstained because the retrieved sources did not provide sufficient information about required computer specifications. Manual inspection suggested that this abstention was appropriate. Subsequent error analysis identified that C25Q01 had been incorrectly classified as a known question and it was therefore reclassified as out-of-KB for the corrected evaluation. This also highlights that automated LLM-based evaluation scores should be interpreted alongside qualitative inspection rather than treated as definitive measures of answer quality.

Generation performance also differed by persona:

**Current Student**
- **Answer Relevancy = 0.708**
- **Faithfulness = 0.917**

**Parent/Guardian**
- **Answer Relevancy = 1.000**
- **Faithfulness = 1.000**

**Prospective Student**
- **Answer Relevancy = 0.875**
- **Faithfulness = 1.000**

Current-student questions showed the lowest Answer Relevancy and Faithfulness within the sample, while parent/guardian questions achieved perfect scores for both metrics. Prospective-student questions achieved perfect Faithfulness with slightly lower Answer Relevancy.

As DeepEval was applied to a stratified sample of 12 questions rather than the full dataset, these results should be interpreted as indicative of generation quality rather than as a definitive estimate of overall system performance.

### Refined Generation Performance

Following the implementation of semantic retrieval, DeepEval was rerun on the corrected stratified sample of 12 questions, with four questions selected from each persona. The refined system achieved:

- **Answer Relevancy = 0.875**
- **Faithfulness = 1.000**

Performance by persona was:

**Current Student**
- **Answer Relevancy = 0.750**
- **Faithfulness = 1.000**

**Parent/Guardian**
- **Answer Relevancy = 1.000**
- **Faithfulness = 1.000**

**Prospective Student**
- **Answer Relevancy = 0.875**
- **Faithfulness = 1.000**

The refined system maintained high Answer Relevancy while achieving a Faithfulness score of 1.000 across all three personas in the evaluated sample. Current-student questions continued to show the lowest Answer Relevancy at 0.750, while parent/guardian questions achieved perfect scores for both metrics.

These results suggest that the retrieval refinement did not reduce generation quality and that the generated answers remained strongly grounded in the retrieved RMIT information. However, as DeepEval was applied to a stratified sample of 12 questions, these results should still be interpreted as indicative rather than as a definitive estimate of overall generation performance.

### Readability

Readability metrics were also calculated for the 12 generated answers. Flesch Reading Ease describes general reading ease, while Flesch-Kincaid Grade Level estimates the educational grade level associated with the text. Average words per sentence and average syllables per word provide additional measures of answer complexity. These metrics describe the linguistic characteristics of the responses and complement the relevance and faithfulness measures.

### Readability Performance

Readability was evaluated on the generated answers using Flesch Reading Ease, Flesch-Kincaid Grade Level, Gunning Fog Index, and SMOG Index, alongside basic answer-length and complexity measures. Flesch Reading Ease provides a measure of reading difficulty where higher scores indicate easier text, while the grade-level measures estimate the education level associated with understanding the response. The Gunning Fog and SMOG measures provide additional estimates based particularly on sentence length and the presence of complex or multi-syllable words.

Because many chatbot answers are relatively short, the SMOG and Gunning Fog results should be interpreted as indicators of linguistic complexity rather than definitive measures of user comprehension. In particular, the standard SMOG formula is designed for longer passages and may be less stable for very short responses. Readability metrics therefore complement, rather than replace, the relevancy and faithfulness measures and qualitative inspection of individual answers.

### Robustness

Robustness was evaluated using all 19 out-of-KB questions and all 51 known questions. The system achieved an Out-of-KB Abstention Rate of 84.2% (16/19 correctly refused) and a False Abstention Rate of 7.8% (4/51 wrongly refused).

Failures were concentrated in predictive questions (e.g. falsely denying any planned new majors) and one knowledge-base gap, C18Q01, where the system fabricated an answer about transferring between Business and Commerce rather than abstaining — unlike the other out-of-KB failures, this is a content gap rather than a robustness limitation. Of the 4 false abstentions, some occurred despite successful retrieval, suggesting the generator's abstention threshold may still be slightly too conservative.

Overall Robustness Accuracy was 90.0%, with a Hallucination Rate of 15.8% on out-of-KB questions.

### Overall Interpretation

Overall, the evaluation demonstrates that the system is capable of producing relevant and well-grounded responses, while the error analysis identified retrieval as an important source of failure in the preliminary BM25 pipeline. Introducing semantic retrieval improved overall Hit@3 from 0.804 to 0.882, Recall@3 from 0.735 to 0.843, and NDCG@3 from 0.706 to 0.781. The largest retrieval improvements occurred for parent/guardian and prospective-student questions.

The refined generation evaluation achieved an Answer Relevancy score of 0.875 and a Faithfulness score of 1.000 on the 12-question DeepEval sample. Robustness evaluation across all 70 questions produced an Out-of-KB Abstention Rate of 84.2%, a False Abstention Rate of 7.8%, and an Overall Robustness Accuracy of 90.0%.

The results also demonstrate the importance of evaluating retrieval and generation separately. Improvements to retrieval increase the availability of relevant evidence, but generation quality, abstention behaviour, and remaining knowledge-base gaps must still be evaluated independently.

Overall, the findings support the use of a multi-layer evaluation framework combining retrieval metrics (Hit@3, Recall@3 and NDCG@3), generation metrics (Answer Relevancy and Faithfulness), readability measures, robustness testing, and manual error analysis. Remaining limitations include retrieval failures concentrated around Bachelor of Games questions, occasional false abstentions and hallucinations on out-of-KB questions, and the limited 12-question sample used for DeepEval.